In [10]:
import numpy as np

# ============================================================
# Basic states and operators
# ============================================================

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

I = np.eye(2, dtype=complex)

X = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

H = (1 / np.sqrt(2)) * np.array([
    [1,  1],
    [1, -1]
], dtype=complex)


def kron(*ops):
    """Kronecker product of multiple operators or states."""
    out = ops[0]
    for op in ops[1:]:
        out = np.kron(out, op)
    return out


# ============================================================
# Two-qubit computational basis
# Ordering: |00>, |01>, |10>, |11>
# ============================================================

basis_labels = ["00", "01", "10", "11"]

ket00 = kron(ket0, ket0)
ket01 = kron(ket0, ket1)
ket10 = kron(ket1, ket0)
ket11 = kron(ket1, ket1)


# ============================================================
# Bell states
# ============================================================

bell_states = {
    "Phi+": (ket00 + ket11) / np.sqrt(2),
    "Phi-": (ket00 - ket11) / np.sqrt(2),
    "Psi+": (ket01 + ket10) / np.sqrt(2),
    "Psi-": (ket01 - ket10) / np.sqrt(2),
}

bell_latex = {
    "Phi+": r"$|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$",
    "Phi-": r"$|\Phi^-\rangle = (|00\rangle - |11\rangle)/\sqrt{2}$",
    "Psi+": r"$|\Psi^+\rangle = (|01\rangle + |10\rangle)/\sqrt{2}$",
    "Psi-": r"$|\Psi^-\rangle = (|01\rangle - |10\rangle)/\sqrt{2}$",
}


# ============================================================
# Bell-state measurement circuit
#
# BSM = inverse Bell preparation
#
# Bell preparation:
#   |ab> -- H on qubit 1 -- CNOT --> Bell state
#
# BSM unitary:
#   Bell state -- CNOT -- H on qubit 1 --> computational basis
# ============================================================

CNOT = np.array([
    [1, 0, 0, 0],   # |00> -> |00>
    [0, 1, 0, 0],   # |01> -> |01>
    [0, 0, 0, 1],   # |10> -> |11>
    [0, 0, 1, 0],   # |11> -> |10>
], dtype=complex)

H_on_first = kron(H, I)

U_BSM = H_on_first @ CNOT


def probabilities(state):
    """Return computational-basis measurement probabilities."""
    return np.abs(state) ** 2


def most_likely_outcome(probs, tol=1e-10):
    """Return deterministic outcome if one probability is 1."""
    idx = int(np.argmax(probs))
    if np.isclose(probs[idx], 1.0, atol=tol):
        return basis_labels[idx]
    return basis_labels[idx]


def run_bsm(bell_name):
    """
    Apply the BSM unitary to a selected Bell state and return all useful data.
    """
    input_state = bell_states[bell_name]
    after_cnot = CNOT @ input_state
    output_state = H_on_first @ after_cnot
    probs = probabilities(output_state)
    outcome = most_likely_outcome(probs)

    return {
        "bell_name": bell_name,
        "input_state": input_state,
        "after_cnot": after_cnot,
        "output_state": output_state,
        "probabilities": probs,
        "outcome": outcome,
    }


def format_state(state, tol=1e-10):
    """
    Format a two-qubit state in computational basis notation.
    """
    terms = []

    for amp, label in zip(state, basis_labels):
        if abs(amp) > tol:
            amp_clean = np.real_if_close(amp)
            terms.append((amp_clean, label))

    if not terms:
        return "0"

    pieces = []
    for amp, label in terms:
        if np.isclose(amp, 1.0):
            pieces.append(f"|{label}>")
        elif np.isclose(amp, -1.0):
            pieces.append(f"-|{label}>")
        elif np.isclose(amp, 1/np.sqrt(2)):
            pieces.append(f"1/sqrt(2)|{label}>")
        elif np.isclose(amp, -1/np.sqrt(2)):
            pieces.append(f"-1/sqrt(2)|{label}>")
        else:
            pieces.append(f"({amp:.3g})|{label}>")

    return " + ".join(pieces).replace("+ -", "- ")


def identify_bell_from_outcome(outcome):
    """
    Deterministic BSM lookup table.
    """
    table = {
        "00": "Phi+",
        "10": "Phi-",
        "01": "Psi+",
        "11": "Psi-",
    }
    return table[outcome]


def print_bsm_summary(bell_name):
    """
    Text summary for notebook output.
    """
    result = run_bsm(bell_name)

    print("Selected Bell state:")
    print(f"  {bell_name}")
    print()
    print("Input state:")
    print(f"  {format_state(result['input_state'])}")
    print()
    print("After CNOT:")
    print(f"  {format_state(result['after_cnot'])}")
    print()
    print("After H ⊗ I:")
    print(f"  {format_state(result['output_state'])}")
    print()
    print("Measurement probabilities:")
    for label, prob in zip(basis_labels, result["probabilities"]):
        print(f"  P({label}) = {prob:.3f}")
    print()
    print("Measured bitstring:")
    print(f"  {result['outcome']}")
    print()
    print("Identified Bell state:")
    print(f"  {identify_bell_from_outcome(result['outcome'])}")

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Math
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Cell 2 — UI + compact text output + live graphics
# Requires Cell 1 tools to be run first
# ============================================================

bell_selector = widgets.Dropdown(
    options=[
        ("|Phi+> = (|00> + |11>)/sqrt(2)", "Phi+"),
        ("|Phi-> = (|00> - |11>)/sqrt(2)", "Phi-"),
        ("|Psi+> = (|01> + |10>)/sqrt(2)", "Psi+"),
        ("|Psi-> = (|01> - |10>)/sqrt(2)", "Psi-"),
    ],
    value="Phi+",
    description="Input:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="520px"),
)

run_button = widgets.Button(
    description="Run BSM",
    button_style="primary",
    layout=widgets.Layout(width="160px"),
)

output_box = widgets.Output()


# ============================================================
# Compact formatting helpers
# ============================================================

def build_compact_state_block(result):
    """
    Left-column text block: selected Bell state and transformation path.
    """
    bell_name = result["bell_name"]

    lines = [
        "Selected Bell state:",
        f"  {bell_name}",
        "",
        "Input state:",
        f"  {format_state(result['input_state'])}",
        "",
        "After CNOT:",
        f"  {format_state(result['after_cnot'])}",
        "",
        "After H ⊗ I:",
        f"  {format_state(result['output_state'])}",
    ]

    return "\n".join(lines)


def build_compact_measurement_block(result):
    """
    Right-column text block: final state, probabilities, measured bitstring,
    and identified Bell state.
    """
    probs = result["probabilities"]
    outcome = result["outcome"]
    identified = identify_bell_from_outcome(outcome)

    lines = [
        "Measurement probabilities:",
    ]

    for label, prob in zip(basis_labels, probs):
        lines.append(f"  P({label}) = {prob:.3f}")

    lines.extend([
        "",
        "Measured bitstring:",
        f"  {outcome}",
        "",
        "Identified Bell state:",
        f"  {identified}",
    ])

    return "\n".join(lines)


def make_pre_widget(text, width="390px"):
    """
    Render fixed-width preformatted text in a widget column.
    """
    return widgets.HTML(
        value=f"""
        <pre style="
            font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
            font-size: 13px;
            line-height: 1.35;
            background: #f7f7f7;
            border: 1px solid #ddd;
            border-radius: 6px;
            padding: 10px;
            margin: 0;
            white-space: pre-wrap;
        ">{text}</pre>
        """,
        layout=widgets.Layout(width=width),
    )


# ============================================================
# Graphics helpers
# ============================================================

def build_bsm_mapping_matrix():
    """
    Return a 4x4 matrix whose rows are input Bell states and columns are
    computational-basis measurement outcomes.
    """
    bell_order = ["Phi+", "Phi-", "Psi+", "Psi-"]
    mapping_matrix = np.zeros((4, 4))

    for i, bell_name in enumerate(bell_order):
        result = run_bsm(bell_name)
        mapping_matrix[i, :] = result["probabilities"]

    return bell_order, mapping_matrix


def plot_bsm_graphics_side_by_side(bell_name):
    """
    Plot two smaller square graphics side-by-side:
      1. Output probabilities for the selected Bell state.
      2. Full deterministic Bell-state measurement map.
    """
    result = run_bsm(bell_name)
    probs = result["probabilities"]

    bell_order, mapping_matrix = build_bsm_mapping_matrix()

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(7.6, 3.4),
        constrained_layout=True,
    )

    # --------------------------------------------------------
    # Left panel: selected Bell-state output probabilities
    # --------------------------------------------------------
    ax = axes[0]
    ax.bar(basis_labels, probs)

    ax.set_box_aspect(1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Measured bitstring", fontsize=9)
    ax.set_ylabel("Probability", fontsize=9)
    ax.set_title(f"BSM output for {bell_name}", fontsize=10)
    ax.tick_params(axis="both", labelsize=8)

    for x, p in zip(basis_labels, probs):
        ax.text(
            x,
            min(p + 0.04, 1.03),
            f"{p:.2f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    # --------------------------------------------------------
    # Right panel: full BSM map
    # --------------------------------------------------------
    ax = axes[1]
    im = ax.imshow(
        mapping_matrix,
        aspect="equal",
        vmin=0,
        vmax=1,
    )

    ax.set_box_aspect(1)
    ax.set_xticks(range(4))
    ax.set_xticklabels(basis_labels, fontsize=8)
    ax.set_yticks(range(4))
    ax.set_yticklabels(bell_order, fontsize=8)

    ax.set_xlabel("Measured bitstring", fontsize=9)
    ax.set_ylabel("Input Bell state", fontsize=9)
    ax.set_title("Full BSM map", fontsize=10)

    for i in range(4):
        for j in range(4):
            ax.text(
                j,
                i,
                f"{mapping_matrix[i, j]:.0f}",
                ha="center",
                va="center",
                fontsize=9,
            )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label("Probability", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    plt.show()


# ============================================================
# UI callback
# ============================================================

def on_run_bsm_clicked(_):
    with output_box:
        clear_output(wait=True)

        bell_name = bell_selector.value
        result = run_bsm(bell_name)

        display(Math(bell_latex[bell_name]))

        left_block = make_pre_widget(
            build_compact_state_block(result),
            width="390px",
        )

        right_block = make_pre_widget(
            build_compact_measurement_block(result),
            width="390px",
        )

        display(
            widgets.HBox(
                [left_block, right_block],
                layout=widgets.Layout(
                    gap="12px",
                    align_items="stretch",
                    margin="0 0 10px 0",
                ),
            )
        )

        display(
            widgets.HTML(
                value="""
                <div style="
                    font-size: 14px;
                    margin: 6px 0 12px 0;
                    padding: 8px 10px;
                    background: #eef6ff;
                    border-left: 4px solid #4477aa;
                    border-radius: 4px;
                ">
                <b>Conclusion:</b>
                The two-bit measurement result identifies the input Bell state
                with 100% certainty.
                </div>
                """
            )
        )

        plot_bsm_graphics_side_by_side(bell_name)


run_button.on_click(on_run_bsm_clicked)


# ============================================================
# Display UI
# ============================================================

display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Bell-State Measurement Simulator</h3>"),
            widgets.HTML(
                "Select one Bell state. The simulator applies the inverse "
                "Bell-preparation circuit and measures both qubits."
            ),
            bell_selector,
            run_button,
            output_box,
        ]
    )
)

# Run once by default
on_run_bsm_clicked(None)